# Thesis Analysis: Fairness in AVMs


In [ ]:
# =================================================
# 1. IMPORTS & SETUP
# =================================================
# This cell imports all necessary libraries and configures the plotting environment.
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import geopandas as gpd
import contextily as cx
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import graphviz
import matplotlib_inline.backend_inline
from IPython.display import display_html, display, Markdown
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
from scipy import stats

# Forces text to be drawn as shapes rather than font tags
plt.rcParams['svg.fonttype'] = 'path'
# --- Plotting Style ---
# Set the global theme and style for all plots in the notebook.
sns.set_theme(style="whitegrid", font_scale=1.3)
# Configure inline backend to save figures in vector formats for high quality.
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')
plt.rcParams['svg.fonttype'] = 'none' # Ensures fonts are not outlined
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['savefig.format'] = 'pdf'
mpl.rcParams['savefig.transparent'] = True

# --- Colorblind-Safe Palette (Okabe-Ito) ---
# Define a colorblind-safe palette for categorical data.
okabe_ito = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2",
                "#D55E00", "#CC79A7", "#000000", "#795548", "#607D8B"]
sns.set_palette(sns.color_palette(okabe_ito))

# --- Okabe-Ito Continuous Colormaps ---
# Create and register diverging and sequential colormaps based on the Okabe-Ito palette.
colors_list = ["#56B4E9", "#F2F2F2", "#D55E00"]
cmap_okabe_div = mcolors.LinearSegmentedColormap.from_list("OkabeDiverging", colors_list)
if "OkabeDiverging" not in mpl.colormaps:
    mpl.colormaps.register(cmap=cmap_okabe_div)

colors_seq = ["#F2F2F2", "#56B4E9", "#0072B2"]
cmap_okabe_seq = mcolors.LinearSegmentedColormap.from_list("OkabeSequential", colors_seq)
if "OkabeSequential" not in mpl.colormaps:
    mpl.colormaps.register(cmap=cmap_okabe_seq)

# --- Academic Formatting Function ---
# Helper function to apply a consistent, academic-style formatting to plot axes.
def enforce_academic_formatting(ax):
    # Apply a light grey grid for better readability.
    ax.grid(True, which='both', axis='both', linestyle='-', alpha=0.3, color='lightgrey')
    ax.set_axisbelow(True)

    # Helper to check if axis is categorical to avoid modifying non-numeric labels.
    def is_categorical(axis):
        labels = [t.get_text() for t in axis.get_ticklabels()]
        for l in labels:
            if l.strip() and not l.replace('.','',1).replace('-','',1).isdigit():
                return True
        return False


In [ ]:
# =================================================
# 2. CONFIGURATION
# =================================================
# This cell defines all file paths and configurations for the analysis runs.

# --- Input Directories ---
# Define base paths for the unweighted (baseline) and weighted clustering results.
BASELINE_DIR = r"C:\Users\kaiso\PycharmProjects\BPAI\clustering_results\Results_Unweighted\s42_Unweighted_Baseline"
WEIGHTED_DIR = r"C:\Users\kaiso\PycharmProjects\BPAI\clustering_results\Results_Weighted\s42_Weighted_Baseline\s42_w_ADI_STATERNK2.0"
CENSUS_SHAPEFILE_PATH = r"C:\Users\kaiso\PycharmProjects\BPAI\Data\shapefile_2015_BlockGroup_Calif\tl_2015_06_bg.shp"

# --- Input Filenames ---
# Standardized filenames for the various output files from the clustering pipeline.
SUMMARY_FILENAME = "results_summary.csv"
SOCIO_SUMMARY_FILENAME = "-reg+SEN+ERR.csv"
GEO_SUMMARY_FILENAME = "+REG+SEN+ERR.csv"
SOCIO_FULL_DATA_FILENAME = "-reg+SEN+ERR_full_data.csv"
GEO_FULL_DATA_FILENAME = "+REG+SEN+ERR_full_data.csv"
SOCIO_COMPOSITION_FILENAME = "-reg+SEN+ERR_composition_ADI_STATERNK_composition_data.csv"
GEO_COMPOSITION_FILENAME = "+REG+SEN+ERR_composition_ADI_STATERNK_composition_data.csv"

# --- Run Configuration ---
# A helper function to create a dictionary of paths for a given analysis run.
def create_run_config(run_dir, name):
    return {
        "name": name,
        "dir": run_dir,
        "summary": os.path.join(run_dir, SUMMARY_FILENAME),
        "socio_summary": os.path.join(run_dir, SOCIO_SUMMARY_FILENAME),
        "geo_summary": os.path.join(run_dir, GEO_SUMMARY_FILENAME),
        "socio_full_data": os.path.join(run_dir, SOCIO_FULL_DATA_FILENAME),
        "geo_full_data": os.path.join(run_dir, GEO_FULL_DATA_FILENAME),
        "socio_composition": os.path.join(run_dir, SOCIO_COMPOSITION_FILENAME),
        "geo_composition": os.path.join(run_dir, GEO_COMPOSITION_FILENAME),
    }

# Define the list of analysis runs to be processed in this notebook.
runs = [
    create_run_config(BASELINE_DIR, "Baseline"),
    create_run_config(WEIGHTED_DIR, "Weighted (ADI x2.0)")
]

# --- Path Verification ---
# Check that all configured file paths exist and print their status.
print("--- Path Configuration Verification ---")
for run in runs:
    print(f"\n--- {run['name']} ---")
    for key, path in run.items():
        if key in ['name', 'dir']: continue
        status = "✅" if os.path.exists(path) else "❌"
        print(f"{status} {key}: {os.path.basename(path)}")
print("\n--- Configuration Complete ---")


## 3. Data Loading and Pre-computation


In [ ]:
# =================================================
# 3.1 DATA LOADING AND PROCESSING
# =================================================
# This cell loads all datasets, performs necessary pre-processing, and caches them in memory.

# A dictionary to hold all loaded dataframes to avoid re-loading from disk.
cached_data = {}

# --- Helper Function: Calculate Weighted ADI ---
# Calculates a weighted average ADI for a cluster based on the proportion of properties from each ADI rank.
def get_weighted_adi(row):
    if 'ADI_STATERNK_avg' in row.index and not pd.isna(row['ADI_STATERNK_avg']):
        return float(row['ADI_STATERNK_avg'])
    adi_cols = [c for c in row.index if 'ADI_STATERNK' in c and c.endswith('_prop')]
    props = row[adi_cols]
    if props.isna().all() or props.sum() == 0: return np.nan
    weighted_sum = 0; total_prop = 0
    for col in adi_cols:
        try:
            rank = int(col.split('=')[1].replace('_prop', ''))
            prop = float(row[col])
            if not np.isnan(prop):
                weighted_sum += rank * prop
                total_prop += prop
        except: pass
    if total_prop == 0: return np.nan
    return weighted_sum / total_prop

# --- Load LA County Shapefile ---
# Load and cache the geographic border of LA County for mapping, removing islands.
try:
    census_gdf = gpd.read_file(CENSUS_SHAPEFILE_PATH)
    la_census_gdf = census_gdf[census_gdf['COUNTYFP'] == '037']

    # Explode to remove islands
    la_exploded = la_census_gdf.explode(index_parts=False)
    mainland_mask = la_exploded.to_crs(epsg=4326).geometry.centroid.y > 33.55
    la_mainland = la_exploded[mainland_mask]

    cached_data['la_border'] = la_mainland.dissolve()
    del census_gdf, la_census_gdf, la_exploded, la_mainland
except Exception as e:
    print(f"❌ CRITICAL: Could not load or process shapefile: {e}")

# --- Load and Process Data for Each Run ---
# Loop through each configured run (Baseline, Weighted) to load and process its data.
for run in runs:
    run_name = run['name']
    print(f"\n--- Processing: {run_name} ---")

    # --- Socioeconomic Data ---
    # Load full property data and cluster summaries for the socioeconomic analysis.
    if os.path.exists(run['socio_full_data']) and os.path.exists(run['socio_summary']):
        df_socio_full = pd.read_csv(run['socio_full_data'])
        df_socio_summary = pd.read_csv(run['socio_summary'])
        df_socio_summary = df_socio_summary[~df_socio_summary['c'].astype(str).str.startswith(('OVERALL', 'SEP'))].copy()

        # Data cleaning and feature engineering.
        for col in ['c', 'count', 'logerror_mean', 'abs_logerror_mean']:
            df_socio_summary[col] = pd.to_numeric(df_socio_summary[col], errors='coerce')
        df_socio_summary['dominant_adi'] = df_socio_summary.apply(get_weighted_adi, axis=1)
        df_socio_summary['rounded_adi'] = df_socio_summary['dominant_adi'].round()

        # Map the calculated dominant ADI back to the full dataset.
        socio_map_df = df_socio_summary.dropna(subset=['dominant_adi', 'c']).copy()
        socio_map_df['c'] = socio_map_df['c'].astype(int)
        cluster_adi_map = socio_map_df.set_index('c')['dominant_adi'].to_dict()

        df_socio_full['cluster_dominant_adi'] = df_socio_full['clusters'].map(cluster_adi_map)
        df_socio_full = df_socio_full.dropna(subset=['cluster_dominant_adi'])
        df_socio_full['abs_logerror'] = df_socio_full['logerror'].abs()
        df_socio_full['rounded_adi'] = df_socio_full['cluster_dominant_adi'].round()

        # Store the processed dataframes in the cache.
        cached_data[f"socio_df_mapped_{run_name}"] = df_socio_full
        cached_data[f"socio_summary_{run_name}"] = df_socio_summary
    else:
        print(f"Skipping socio data for '{run_name}' (files not found).")

    # --- Geographic Data ---
    # Load full property data and cluster summaries for the geographic analysis.
    if os.path.exists(run['geo_full_data']) and os.path.exists(run['geo_summary']):
        df_geo_full = pd.read_csv(run['geo_full_data'])
        df_geo_summary = pd.read_csv(run['geo_summary'])
        df_geo_summary = df_geo_summary[~df_geo_summary['c'].astype(str).str.startswith(('OVERALL', 'SEP'))].copy()

        # Data cleaning and feature engineering.
        for col in ['c', 'count', 'logerror_mean', 'abs_logerror_mean', 'diff_vs_rest']:
            df_geo_summary[col] = pd.to_numeric(df_geo_summary[col], errors='coerce')
        df_geo_summary['dominant_adi'] = df_geo_summary.apply(get_weighted_adi, axis=1)
        df_geo_summary['rounded_adi'] = df_geo_summary['dominant_adi'].round()

        # Map the calculated dominant ADI back to the full dataset.
        geo_map_df = df_geo_summary.dropna(subset=['dominant_adi', 'c']).copy()
        geo_map_df['c'] = geo_map_df['c'].astype(int)
        cluster_adi_map = geo_map_df.set_index('c')['dominant_adi'].to_dict()

        df_geo_full['cluster_dominant_adi'] = df_geo_full['clusters'].map(cluster_adi_map)
        df_geo_full = df_geo_full.dropna(subset=['cluster_dominant_adi'])

        # Convert the geographic dataframe to a GeoDataFrame for spatial analysis.
        gdf_geo_full = gpd.GeoDataFrame(
            df_geo_full,
            geometry=gpd.points_from_xy(df_geo_full.longitude, df_geo_full.latitude),
            crs="EPSG:4326"
        )
        cached_data[f"geo_gdf_mapped_{run_name}"] = gdf_geo_full
        cached_data[f"geo_summary_{run_name}"] = df_geo_summary
    else:
        print(f"Skipping geo data for '{run_name}' (files not found).")

    # --- Composition Data ---
    # Load the cluster composition data (proportion of each ADI rank in each cluster).
    if os.path.exists(run['socio_composition']):
        cached_data[f"socio_composition_{run_name}"] = pd.read_csv(run['socio_composition'])
    if os.path.exists(run['geo_composition']):
        cached_data[f"geo_composition_{run_name}"] = pd.read_csv(run['geo_composition'])

print("\n--- Pre-computation Complete ---")

### Presentation Mode Configuration & Tips

To improve the visibility of graphs from a distance for presentations, you can run the following code cell. It will globally increase font sizes, line widths, and marker sizes for all subsequent plots in this session. To revert to the default academic style, simply re-run the setup cell (`1. IMPORTS & SETUP`).

**Additional Tips for Visibility:**

*   **Distinct Markers:** For scatter plots with multiple categories, use distinct markers (e.g., circles, squares, triangles) in addition to color. This helps distinguish categories even if colors are hard to see. The plot in section `12.1` is a great example of this.
*   **High-Contrast Colors:** The Okabe-Ito palette is excellent. For emphasis, you can manually select the most contrasting colors from the palette for key data points (e.g., black, orange, blue).
*   **Simplify:** For a presentation, consider removing less critical annotations or grid lines to reduce clutter and focus the audience's attention.


In [ ]:
# =================================================
# PRESENTATION MODE SETTINGS
# =================================================
# This cell globally increases plot element sizes for better visibility in presentations.

print("--- Activating Presentation Mode ---")
# Increase the base font scale for all text elements
sns.set_theme(style="whitegrid", font_scale=2.0)
# Update matplotlib's rcParams for elements not controlled by seaborn's theme
mpl.rcParams.update({
    'lines.linewidth': 4,       # Thicker lines in line plots
    'lines.markersize': 12,     # Bigger markers
    'axes.labelsize': 28,       # X and Y axis labels
    'xtick.labelsize': 24,      # X-axis tick labels
    'ytick.labelsize': 24,      # Y-axis tick labels
    'legend.fontsize': 22,      # Legend text
    'legend.title_fontsize': 24,# Legend title
    'figure.titlesize': 32      # Figure-level titles (if any)
})
print("Global plot styles updated for presentations. Re-run any plotting cell to see the changes.")
# Re-apply the colorblind-safe palette as set_theme might reset it
sns.set_palette(sns.color_palette(okabe_ito))



## 4. Initial Data Exploration


In [ ]:
# =================================================
# 4.1 GLOBAL METRICS & DEPRIVATION DISTRIBUTION
# =================================================
# This cell calculates overall model performance metrics and visualizes the property distribution across ADI ranks.
df = cached_data.get("socio_df_mapped_Baseline")

if df is not None:
    # 1. Global Model Metrics (Bias and Inaccuracy)
    # Calculate the mean logerror (overall bias) and mean absolute logerror (overall inaccuracy).
    global_logerror_avg = df['logerror'].mean()
    global_abs_logerror_avg = df['logerror'].abs().mean()

    print("--- Global Model Metrics (Pre-Clustering) ---")
    print(f"Average Log Error (Bias): {global_logerror_avg:.4f}")
    print(f"Average Absolute Log Error (Inaccuracy): {global_abs_logerror_avg:.4f}")
    print("-" * 45)

    # 2. Kruskal-Wallis Test across Clusters
    # This test determines if there are statistically significant differences in error distributions between clusters.
    if 'clusters' in df.columns:
        # Group logerrors by cluster membership
        cluster_groups = [group['logerror'].values for _, group in df.groupby('clusters')]
        h_stat, p_value = stats.kruskal(*cluster_groups)

        print(f"Kruskal-Wallis H-statistic: {h_stat:.4f}")
        # Format p-value for cases where it's extremely small.
        formatted_p = "p < 2.2 x 10^-16" if p_value == 0.0 else f"{p_value}"
        print(f"Kruskal-Wallis p-value: {formatted_p}")
        print("-" * 45)
    else:
        print("⚠️ 'clusters' column not found in dataframe. Skipping Kruskal-Wallis test.")

    # 3. Property Distribution Plot
    # Visualize the number of properties in each socioeconomic (ADI) rank.
    plt.figure(figsize=(14, 8))
    ax = sns.countplot(x='ADI_STATERNK', data=df, hue='ADI_STATERNK', legend=False, palette=okabe_ito)

    plt.xlabel('Area Deprivation Index (1 = Most Affluent, 10 = Most Deprived)', fontsize=26, labelpad=25)
    plt.ylabel('Number of Properties', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.get_yaxis().set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ",")))

    # Apply consistent academic formatting and save the figure.
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('adi_distribution.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('adi_distribution.svg', format='svg', bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Baseline data not found. Cannot calculate global metrics.")


In [ ]:
# =================================================
# 4.2 PROPERTY COUNT BY ADI RANK
# =================================================
# This cell creates a bar plot showing the exact number of properties for each ADI rank.
df_full = cached_data.get("socio_df_mapped_Baseline")

if df_full is not None:
    # Count properties in each ADI rank.
    adi_counts = df_full['ADI_STATERNK'].value_counts().sort_index()
    plt.figure(figsize=(15, 8))
    ax = sns.barplot(x=adi_counts.index, y=adi_counts.values, hue=adi_counts.index, legend=False, palette=okabe_ito)

    # Formatting labels and ticks.
    plt.xlabel('Area Deprivation Index (1 = Most Affluent, 10 = Most Deprived)', fontsize=26, labelpad=25)
    plt.ylabel('Number of Properties', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.get_yaxis().set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))

    # Annotate each bar with its count.
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom',
                    fontsize=24, fontweight='bold',
                    xytext=(0, 5), textcoords='offset points')

    plt.ylim(0, max(adi_counts.values) * 1.1)
    plt.tight_layout(pad=2.0)

    # Apply academic formatting and save.
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('sample_size_verification.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('sample_size_verification.svg', format='svg', bbox_inches='tight')
    plt.show()
else:
    print("⚠️ Baseline data not found. Cannot generate sample size plot.")


In [ ]:
# =================================================
# 4.3 LOGERROR VOLATILITY BY ADI RANK
# =================================================
# This cell creates a boxplot to visualize the spread (volatility) of valuation errors across ADI ranks.

# --- Configuration ---
TARGET_RUN_NAME = "Baseline"
# --- End Configuration ---

# Find the path to the full data file for the target run.
data_path = None
for run in runs:
    if run['name'] == TARGET_RUN_NAME:
        data_path = run['socio_full_data']
        break

if data_path and os.path.exists(data_path):
    # Load the full dataset.
    df_full = pd.read_csv(data_path)
    df_full = df_full[df_full['ADI_STATERNK'].notna()].copy()

    # Create the boxplot.
    plt.figure(figsize=(16, 10))
    ax = sns.boxplot(data=df_full, x='ADI_STATERNK', y='logerror', hue='ADI_STATERNK', legend=False, fliersize=2, palette=okabe_ito)

    # Formatting labels and ticks.
    plt.xlabel('ADI State Rank (1 = Least Deprived, 10 = Most Deprived)', fontsize=26, labelpad=25)
    plt.ylabel('Logerror (Positive = Overvaluation)', fontsize=26, labelpad=25)
    plt.tick_params(axis='y', labelsize=24)
    plt.tick_params(axis='x', labelsize=24)
    plt.axhline(0, ls='-', color='black', linewidth=3) # Zero-error line

    # Improve readability by clipping the y-axis to focus on the interquartile ranges.
    Q1 = df_full['logerror'].quantile(0.25)
    Q3 = df_full['logerror'].quantile(0.75)
    IQR = Q3 - Q1
    y_limit = (Q1 - 2.5 * IQR, Q3 + 2.5 * IQR)
    plt.ylim(y_limit)

    plt.tight_layout(pad=2.0)
    # Apply academic formatting and save.
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('logerror_volatility.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('logerror_volatility.svg', format='svg', bbox_inches='tight')
    plt.show()

else:
    print(f"Could not find or process data for run: '{TARGET_RUN_NAME}'")


In [ ]:
# =================================================
# 4.4 ERROR DENSITY HEXBIN PLOT
# =================================================
# This cell creates a hexbin plot to show the density of properties at different levels of error and deprivation.

# --- 0. Data Input ---
property_df = cached_data.get("socio_df_mapped_Baseline")

# --- Guard Clause ---
if property_df is None or 'ADI_STATERNK' not in property_df.columns or 'abs_logerror' not in property_df.columns:
    print("⚠️ Prerequisite data not found. Please run cell 3.1 to load and process the data.")
else:
    # --- 1. Figure Setup ---
    fig, ax = plt.subplots(figsize=(12, 9))

    # Crop the Y-axis to zoom in on the core density of the data.
    ymin = 0.0
    ymax = 0.5

    # --- 2. Hexbin Generation & Grid Sizing ---
    # The extent and gridsize are set to align hexagon centers with integer ADI ranks.
    extent = [0.5, 10.5, ymin, ymax]
    gridsize = (10, 15) # 10 bins horizontally, 15 vertically

    hb = ax.hexbin(
        x=property_df['ADI_STATERNK'],
        y=property_df['abs_logerror'],
        gridsize=gridsize,
        extent=extent,
        cmap='OkabeSequential', # Colorblind-safe, sequential colormap
        mincnt=1, # Don't draw empty hexagons
        linewidths=0.2,
        edgecolors='white'
    )

    # --- 3. Formatting & Aesthetics ---
    ax.set_xlabel('Area Deprivation Index (1 = Affluent, 10 = Deprived)', fontsize=26, labelpad=27)
    ax.set_ylabel('Absolute Logerror (Magnitude of Inaccuracy)', fontsize=26, labelpad=27)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.set_xticks(range(1, 11)) # Ensure discrete integer ticks for ADI ranks
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Add and format the colorbar.
    cbar = fig.colorbar(hb, ax=ax)
    cbar.set_label('Number of Properties in Bin', fontsize=26, labelpad=15)
    cbar.ax.tick_params(labelsize=24)

    # --- 4. Export & Display ---
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('hexbin_bias_density.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('hexbin_bias_density.svg', format='svg', bbox_inches='tight')
    plt.show()


## 5. Statistical Significance of Clusters


In [ ]:
# =================================================
# 5.1 KRUSKAL-WALLIS TEST FOR CLUSTERS
# =================================================
# This cell checks the summary file for each run to confirm that the clusters have statistically distinct error distributions.
for run in runs:
    print(f"\n--- {run['name']} ---")
    summary_path = run['summary']
    # Fallback for slightly different summary filenames.
    if not os.path.exists(summary_path):
        files = [f for f in os.listdir(run['dir']) if f.startswith('results_summary_') and f.endswith('.csv')]
        if files:
            summary_path = os.path.join(run['dir'], files[0])
            run['summary'] = summary_path

    # Read the summary and display key results.
    if os.path.exists(summary_path):
        summary_df = pd.read_csv(summary_path)
        cond_col = 'condition' if 'condition' in summary_df.columns else summary_df.columns[0]
        display_cols = [c for c in [cond_col, 'n_clusters', 'kw_p_logerror', 'logerror_avg', 'abs_logerror_avg'] if c in summary_df.columns]
        display(summary_df[display_cols])

        # Check if the p-value from the Kruskal-Wallis test is below the significance level (0.05).
        if 'kw_p_logerror' in summary_df.columns and (summary_df['kw_p_logerror'] < 0.05).all():
            print("\n✅ SUCCESS: Clusters show statistically distinct error distributions (p < 0.05).")
        else:
            print("\n⚠️ WARNING: Some clusters do not show statistically significant error distributions.")
    else:
        print(f"❌ Summary file not found in '{run['dir']}'")


## 6. Cluster Composition & Isolation Analysis


In [ ]:
# =================================================
# 6.1 CLUSTER COMPOSITION PLOTS
# =================================================
# This cell defines a function to plot the socioeconomic (ADI) breakdown of each cluster.

def plot_cluster_composition(run_name, composition_df, summary_df, title_prefix, file_prefix):
    """Generates a stacked bar chart showing the ADI composition of each cluster."""
    if composition_df is None:
        print(f"❌ No composition data for {run_name} ({title_prefix})")
        return
    if summary_df is None:
        print(f"❌ No summary data for {run_name} ({title_prefix})")
        return

    # Sort clusters by their dominant ADI for a clean visual progression.
    dominant_adi_map = summary_df.set_index('c')['dominant_adi']
    composition_df = composition_df.copy()
    composition_df['dominant_adi'] = composition_df['cluster'].map(dominant_adi_map)

    df_sorted = composition_df.dropna(subset=['dominant_adi']).sort_values(by='dominant_adi').reset_index(drop=True)
    if df_sorted.empty:
        print(f"❌ No valid clusters to plot for {run_name}")
        return

    # Prepare data for stacked bar plot.
    composition_to_plot = df_sorted.drop(columns=['dominant_adi']).set_index('cluster')
    prop_cols = [f'prop_{i}' for i in range(1, 11)]
    composition_to_plot = composition_to_plot[prop_cols]
    composition_to_plot.columns = range(1, 11)
    composition_to_plot = composition_to_plot * 100 # Convert proportions to percentages

    # Create the plot.
    fig, ax = plt.subplots(figsize=(20, 8), layout="constrained")
    cb_palette = sns.color_palette(okabe_ito, 10)
    composition_to_plot.plot(kind='bar', stacked=True, color=cb_palette, ax=ax, width=1.0)

    # Formatting and labels.
    ax.set_xlabel('Socioeconomic Cluster Rank', fontsize=26, labelpad=25)
    ax.set_ylabel('Proportion (%)', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24, rotation=0)
    ax.set_yticklabels(['{:,.0f}%'.format(x) for x in ax.get_yticks()])
    ax.legend(title='ADI State Rank', bbox_to_anchor=(1.0, 1), loc='upper left', borderaxespad=0., fontsize=24, title_fontsize=26, frameon=True, edgecolor='black', framealpha=1)

    # Apply academic formatting and save.
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f'{file_prefix}_{run_name.replace(" ", "_")}.pdf', format='pdf', bbox_inches='tight')
    plt.savefig(f'{file_prefix}_{run_name.replace(" ", "_")}.svg', format='svg', bbox_inches='tight')
    plt.show()
    plt.close(fig)

# --- Generate Plots for Each Run ---
print("Generating Cluster Composition Plots...")
for run in runs:
    # Plot for geographic clusters.
    plot_cluster_composition(
        run_name=run['name'],
        composition_df=cached_data.get(f"geo_composition_{run['name']}"),
        summary_df=cached_data.get(f"geo_summary_{run['name']}"),
        title_prefix='Geographic',
        file_prefix='composition_geo'
    )
    # Plot for socioeconomic clusters.
    plot_cluster_composition(
        run_name=run['name'],
        composition_df=cached_data.get(f"socio_composition_{run['name']}"),
        summary_df=cached_data.get(f"socio_summary_{run['name']}"),
        title_prefix='Socioeconomic',
        file_prefix='composition_socio'
    )
print("Done.")


In [ ]:
# =================================================
# 6.2 CLUSTER ISOLATION VIOLIN PLOTS
# =================================================
# This cell visualizes cluster isolation, showing how well each cluster groups properties with a similar ADI.
def plot_violin(data_path, ax, title):
    """Generates a violin plot showing the distribution of ADI ranks within each cluster."""
    if os.path.exists(data_path):
        df = pd.read_csv(data_path)
        df = df[df['clusters'] != -1].copy() # Exclude noise points

        # Sort clusters by their median ADI for a clean visual progression.
        cluster_order = df.groupby('clusters')['ADI_STATERNK'].median().sort_values().index

        # Create the violin plot.
        sns.violinplot(data=df, x='clusters', y='ADI_STATERNK', hue='clusters', order=cluster_order, inner='quartile', legend=False, ax=ax, palette=okabe_ito)

        # Formatting and labels.
        ax.set_xlabel('Socioeconomic Cluster Rank', fontsize=26, labelpad=25)
        ax.set_ylabel('Median Area Deprivation Index (ADI)', fontsize=26, labelpad=25)
        ax.tick_params(axis='y', labelsize=24)
        ax.tick_params(axis='x', labelsize=24)
    else:
        ax.set_title(f"Data missing: {os.path.basename(data_path)}", fontsize=24, color='red')

# --- Generate Individual Plots for Each Run and Cluster Type ---
for i, run in enumerate(runs):
    # Socioeconomic clusters
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    plot_violin(run['socio_full_data'], ax, f"[{run['name']}] Isolation (-reg+SEN+ERR)")
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f"socioeconomic_density_isolation_{run['name']}.pdf", format='pdf', bbox_inches='tight')
    plt.savefig(f"socioeconomic_density_isolation_{run['name']}.svg", format='svg', bbox_inches='tight')
    plt.show()

    # Geographic clusters
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    plot_violin(run['geo_full_data'], ax, f"[{run['name']}] Geographic (+REG+SEN+ERR)")
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f"socioeconomic_density_geographic_{run['name']}.pdf", format='pdf', bbox_inches='tight')
    plt.savefig(f"socioeconomic_density_geographic_{run['name']}.svg", format='svg', bbox_inches='tight')
    plt.show()


In [ ]:
# =================================================
# 6.3 UNIFIED CLUSTER PROFILE PLOTS
# =================================================
# This cell creates a two-part plot for each cluster: a stacked bar chart of its composition and a violin plot of its ADI distribution.
def plot_unified_cluster_profile(run_name, violin_data_path, composition_df, file_prefix):
    """Generates a detailed, two-part profile for each cluster."""
    if not os.path.exists(violin_data_path) or composition_df is None:
        print(f"❌ Missing data for unified plot: {run_name}")
        return

    # 1. Load and process violin plot data (individual properties).
    df_violin = pd.read_csv(violin_data_path)
    df_violin = df_violin[df_violin['clusters'] != -1].copy()

    # Lock the global cluster order by median ADI.
    cluster_order = df_violin.groupby('clusters')['ADI_STATERNK'].median().sort_values().index.tolist()

    # 2. Process stacked bar data using the same cluster order.
    comp_df = composition_df.copy().set_index('cluster')
    comp_df = comp_df.reindex(cluster_order)  # Force sequence lock

    prop_cols = [f'prop_{i}' for i in range(1, 11)]
    comp_to_plot = comp_df[prop_cols] * 100
    comp_to_plot.columns = range(1, 11)

    # Prepare descriptive labels for the cluster ranks.
    rank_labels = [f"Rank {i+1}" for i in range(len(cluster_order))]
    if len(cluster_order) > 0:
        rank_labels[0] += "\\n(Most Affluent)"
        rank_labels[-1] += "\\n(Most Deprived)"

    # --- TOP PLOT: Stacked Bar Chart (Proportions) ---
    fig_bar, ax_bar = plt.subplots(figsize=(18, 7))
    cb_palette = sns.color_palette(okabe_ito, 10)
    comp_to_plot.plot(kind='bar', stacked=True, color=cb_palette, ax=ax_bar, width=0.85)

    ax_bar.set_ylabel('Proportion (%)', fontsize=26, labelpad=25)
    ax_bar.tick_params(axis='y', labelsize=24)
    ax_bar.set_yticklabels(['{:,.0f}%'.format(x) for x in ax_bar.get_yticks()])
    ax_bar.legend(title='ADI State Rank', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=24, title_fontsize=26, frameon=True, edgecolor='black', framealpha=1)

    ax_bar.set_ylim(bottom=0)
    ax_bar.set_xticks(range(len(cluster_order)))
    ax_bar.set_xticklabels(rank_labels, fontsize=24)
    ax_bar.set_xlabel('Cluster Rank (Ascending ADI)', fontsize=26, labelpad=25)

    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f"profile_{file_prefix}_bar_{run_name.replace(' ', '_')}.pdf", bbox_inches='tight')
    plt.savefig(f"profile_{file_prefix}_bar_{run_name.replace(' ', '_')}.svg", bbox_inches='tight')
    plt.show()
    plt.close(fig_bar)

    # --- BOTTOM PLOT: Violin Plot (Densities) ---
    fig_violin, ax_violin = plt.subplots(figsize=(18, 7))
    # A neutral color is used as the composition is shown in the bar chart.
    neutral_color = "#607D8B"
    sns.violinplot(data=df_violin, x='clusters', y='ADI_STATERNK',
                   order=cluster_order, color=neutral_color, inner='quartile', ax=ax_violin)

    ax_violin.set_xlabel('Cluster Rank (Ascending ADI)', fontsize=26, labelpad=25)
    ax_violin.set_ylabel('Median Area Deprivation Index (ADI)', fontsize=26, labelpad=25)

    # Enforce y-axis to show every tick from 1 to 10.
    ax_violin.set_yticks(range(1, 11))
    ax_violin.set_ylim(0.5, 10.5)

    ax_violin.tick_params(axis='y', labelsize=24)
    ax_violin.set_xticks(range(len(cluster_order)))
    ax_violin.set_xticklabels(rank_labels, fontsize=24)

    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f"profile_{file_prefix}_violin_{run_name.replace(' ', '_')}.pdf", bbox_inches='tight')
    plt.savefig(f"profile_{file_prefix}_violin_{run_name.replace(' ', '_')}.svg", bbox_inches='tight')
    plt.show()
    plt.close(fig_violin)


## 7. Socioeconomic Bias Analysis


In [ ]:
# =================================================
# 7.1 SOCIOECONOMIC CLUSTER SUMMARY
# =================================================
# This cell displays a summary table of socioeconomic clusters for each run.
html_str = ""
for run in runs:
    df_socio_summary = cached_data.get(f"socio_summary_{run['name']}")
    if df_socio_summary is not None:
        # Group clusters by their rounded dominant ADI.
        adi_summary = df_socio_summary.dropna(subset=['rounded_adi']).groupby('rounded_adi').agg(
            total_properties=('count', 'sum'),
            num_clusters=('count', 'count')
        ).reset_index()
        html_str += f"<div style='display:inline-block; margin-right:20px;'><h3 align='center'>{run['name']}</h3>{adi_summary.to_html(index=False)}</div>"
if html_str: display_html(html_str, raw=True)

# =================================================
# 7.2 VISUALIZING SOCIOECONOMIC BIAS
# =================================================
# This cell generates plots to visualize U-shaped bias and directional harm.
for run in runs:
    name = run['name']
    raw_socio = cached_data.get(f"socio_df_mapped_{name}")
    df_socio = cached_data.get(f"socio_summary_{name}")

    if raw_socio is None or df_socio is None:
        print(f"Skipping plots for {name} due to missing data.")
        continue

    # --- Absolute Error (U-Shape) ---
    # This plot shows if properties in extreme socioeconomic areas face larger errors.
    fig, ax = plt.subplots(1, 1, figsize=(13, 8))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sns.pointplot(data=raw_socio, x='rounded_adi', y='abs_logerror', estimator=np.median, errorbar=('ci', 95), capsize=.2, color=okabe_ito[5], ax=ax, scale=0.7, errwidth=2.5)
    ax.set_xlabel('Dominant ADI (1 = Most Affluent, 10 = Most Deprived)', fontsize=26, labelpad=25)
    ax.set_ylabel('Median Absolute Error', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.set_ylim(bottom=0)
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f'socioeconomic_abs_error_{name}.pdf', format='pdf', bbox_inches='tight')
    plt.savefig(f'socioeconomic_abs_error_{name}.svg', format='svg', bbox_inches='tight')
    plt.show()

    # --- Directional Error (Harm) ---
    # This plot shows if vulnerable areas are systematically overvalued and affluent areas undervalued.
    fig, ax = plt.subplots(1, 1, figsize=(13, 8))
    sns.scatterplot(data=df_socio, x='dominant_adi', y='logerror_mean', s=80, alpha=0.7, ax=ax, hue='logerror_mean', legend=False, palette='OkabeDiverging')
    # Calculate and plot a weighted trend line.
    adi_summary = df_socio.dropna(subset=['rounded_adi']).groupby('rounded_adi').agg(
        weighted_logerror=('logerror_mean', lambda x: np.average(x, weights=df_socio.loc[x.index, 'count'])),
    ).reset_index().rename(columns={'weighted_logerror': 'logerror_mean'})
    sns.lineplot(data=adi_summary, x='rounded_adi', y='logerror_mean', ax=ax, color='black', linewidth=2, label='Weighted Macro Trend')
    ax.legend(fontsize=20, frameon=True, edgecolor='black', framealpha=1)
    ax.axhline(0, ls='--', color='black', linewidth=2)
    ax.set_xlabel('Dominant ADI (1 = Most Affluent, 10 = Most Deprived)', fontsize=26, labelpad=25)
    ax.set_ylabel('Mean Logerror (Positive = Overvalued)', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24)
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f'socioeconomic_directional_error_{name}.pdf', format='pdf', bbox_inches='tight')
    plt.savefig(f'socioeconomic_directional_error_{name}.svg', format='svg', bbox_inches='tight')
    plt.show()


## 8. Geographic Bias Analysis


In [ ]:
# =================================================
# 8.1 GEOGRAPHIC CLUSTER SUMMARY
# =================================================
# This cell displays tables of the most overvalued and undervalued geographic clusters.
html_str = ""
display_cols = ['c', 'rule', 'dominant_adi', 'logerror_mean', 'diff_vs_rest', 'abs_logerror_mean']
for run in runs:
    df_geo_sorted = cached_data.get(f"geo_summary_{run['name']}")
    if df_geo_sorted is None:
        html_str += f"<div style='display:inline-block; margin-right:20px;'><h3>{run['name']}</h3><p>Data missing</p></div>"
        continue

    # Sort clusters by mean logerror to find extremes.
    df_geo_sorted = df_geo_sorted.sort_values(by='logerror_mean', ascending=False)
    num_clusters = len(df_geo_sorted)
    if num_clusters == 0:
        table_html = "<p>No valid geographic clusters found.</p>"
    else:
        # Create HTML tables for the top 5 overvalued and undervalued clusters.
        table_html = f"<b>Top 5 Overvalued:</b><br>{df_geo_sorted[display_cols].head(5).to_html(index=False)}<br>"
        table_html += f"<b>Top 5 Undervalued:</b><br>{df_geo_sorted[display_cols].tail(5).to_html(index=False)}"
    html_str += f"<div style='display:inline-block; vertical-align:top; margin-right:20px; max-width: 48%;'><h3 align='center'>{run['name']}</h3>{table_html}</div>"

if html_str: display_html(html_str, raw=True)

# =================================================
# 8.2 VISUALIZING GEOGRAPHIC BIAS
# =================================================
# This cell creates a boxplot showing the error variance for each geographic cluster.
for run in runs:
    name = run['name']
    raw_geo = cached_data.get(f"geo_gdf_mapped_{name}")
    df_geo = cached_data.get(f"geo_summary_{name}")

    if raw_geo is None or df_geo is None:
        print(f"Skipping plots for {name} due to missing data.")
        continue

    # Order clusters by dominant ADI and highlight the most extreme ones.
    ordered_clusters = df_geo.sort_values('dominant_adi')['c'].tolist()
    overvalued_cids = df_geo[df_geo['logerror_mean'] > 0].sort_values('logerror_mean', ascending=False).head(3)['c'].astype(int).tolist()
    undervalued_cids = df_geo[df_geo['logerror_mean'] < 0].sort_values('logerror_mean', ascending=True).head(3)['c'].astype(int).tolist()
    accessibility_colors = {c: okabe_ito[5] if c in overvalued_cids else (okabe_ito[4] if c in undervalued_cids else 'lightgray') for c in ordered_clusters}

    # Create the boxplot.
    fig, ax = plt.subplots(1, 1, figsize=(20, 9))
    sns.boxplot(data=raw_geo, x='clusters', y='logerror', hue='clusters', order=ordered_clusters, palette=accessibility_colors, fliersize=2, legend=False, ax=ax)
    ax.axhline(0, ls='-', color='black', linewidth=3)
    ax.set_xlabel('Socioeconomic Cluster Rank', fontsize=26, labelpad=25)
    ax.set_ylabel('Property Logerror', fontsize=26, labelpad=25)
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.tick_params(axis='x')
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig(f'geographic_error_variance_{name}.pdf', format='pdf')
    plt.savefig(f'geographic_error_variance_{name}.svg', format='svg')
    plt.show()


In [ ]:
# =================================================
# 8.3 RAINCLOUD PLOT OF EXTREME CLUSTERS
# =================================================
# This cell creates a raincloud plot to compare the error distributions of the single most overvalued and undervalued geographic clusters.
import matplotlib.pyplot as plt
import pandas as pd
import ptitprince as pt

# --- 0. Data Input & Preparation ---
TARGET_RUN_NAME = "Weighted (ADI x2.0)"
df_geo_summary = cached_data.get(f"geo_summary_{TARGET_RUN_NAME}")
gdf_full_data = cached_data.get(f"geo_gdf_mapped_{TARGET_RUN_NAME}")

# --- Guard Clause ---
if df_geo_summary is None or gdf_full_data is None:
    print(f"⚠️ Prerequisite data for '{TARGET_RUN_NAME}' not found. Please ensure preceding cells have run successfully.")
else:
    # --- 1. Identify Extreme Clusters ---
    # Find the clusters with the highest (most overvalued) and lowest (most undervalued) mean logerror.
    overvalued_cluster_info = df_geo_summary.loc[df_geo_summary['logerror_mean'].idxmax()]
    undervalued_cluster_info = df_geo_summary.loc[df_geo_summary['logerror_mean'].idxmin()]

    over_id = int(overvalued_cluster_info['c'])
    under_id = int(undervalued_cluster_info['c'])

    # --- 2. Prepare Data for Plotting ---
    # Create descriptive labels and filter the full dataset for these two clusters.
    over_label = f"Most Overvalued (C{over_id}, ADI: {overvalued_cluster_info['dominant_adi']:.1f})"
    under_label = f"Most Undervalued (C{under_id}, ADI: {undervalued_cluster_info['dominant_adi']:.1f})"

    over_df = gdf_full_data[gdf_full_data['clusters'] == over_id].copy()
    over_df['cluster_id'] = over_label
    under_df = gdf_full_data[gdf_full_data['clusters'] == under_id].copy()
    under_df['cluster_id'] = under_label

    # Combine into a single dataframe for plotting.
    extreme_clusters_df = pd.concat([over_df, under_df])

    # Create a color palette that maps labels to specific Okabe-Ito colors.
    categories = list(extreme_clusters_df['cluster_id'].unique())
    color_list = []
    for cat in categories:
        if 'Overvalued' in str(cat):
            color_list.append("#D55E00") # Vermilion
        else:
            color_list.append("#56B4E9") # Sky Blue

    # --- 3. Figure Setup ---
    fig, ax = plt.subplots(figsize=(14, 9))

    # --- 4. Raincloud Generation ---
    # This function creates the half-violin (density), boxplot (quartiles), and stripplot (raw data).
    pt.RainCloud(
        x='cluster_id',         # Categorical variable
        y='logerror',           # Numeric variable
        data=extreme_clusters_df,
        palette=color_list,
        orient='h',
        alpha=0.6,
        width_viol=0.7,
        width_box=0.15,
        point_size=2.5,
        ax=ax
    )

    # --- 5. Formatting & Aesthetics ---
    ax.axvline(x=0.0, color='black', linestyle='--', linewidth=1.5, zorder=0) # Zero-error baseline
    ax.set_xlabel('Directional Logerror (Bias)', fontsize=26, labelpad=30)
    ax.set_ylabel('') # Y-axis labels are descriptive enough.
    ax.tick_params(axis='both', which='major', labelsize=24)
    ax.xaxis.grid(True, linestyle='-', alpha=0.3, color='lightgrey')
    ax.set_axisbelow(True)

    # --- 6. Export & Display ---
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
        _ax.yaxis.grid(False) # Turn off horizontal grid lines for this plot
    plt.savefig('directional_harm_raincloud.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('directional_harm_raincloud.svg', format='svg', bbox_inches='tight')
    plt.show()


## 9. Mapping Geographic Hotspots


In [ ]:
# =================================================
# 9.3 SPATIAL BUBBLE MAP
# =================================================
# This cell creates a bubble map where each bubble is a cluster, its size is the number of properties, and its color is the mean error.
import matplotlib.colors as mcolors
import geopandas as gpd

# --- 1. Base Map Preparation ---
# Load the pre-processed LA County border from cache.
la_county_base = cached_data.get('la_border')
if la_county_base is not None:
    la_county_base_proj = la_county_base.to_crs(epsg=3857)
else:
    print("⚠️ LA County border not found in cache. Map will be drawn without it.")
    la_county_base_proj = gpd.GeoDataFrame(geometry=[], crs="epsg:3857")

# --- 2. Bubble Data Preparation ---
full_geo_df = cached_data.get("geo_gdf_mapped_Weighted (ADI x2.0)")

if full_geo_df is not None:
    # Calculate the geographic center (medoid) of each cluster.
    medoids = full_geo_df.groupby('clusters')[['longitude', 'latitude']].median().reset_index()

    # Merge with summary data to get property count and mean logerror for each cluster.
    df_geo_summary = cached_data.get("geo_summary_Weighted (ADI x2.0)")
    bubble_data = medoids.merge(df_geo_summary, left_on='clusters', right_on='c')

    # Convert to a GeoDataFrame for plotting.
    gdf_clusters = gpd.GeoDataFrame(
        bubble_data,
        geometry=gpd.points_from_xy(bubble_data.longitude, bubble_data.latitude),
        crs="EPSG:4326"
    )
    gdf_clusters_proj = gdf_clusters.to_crs(epsg=3857)
else:
    print("⚠️ Required data for bubble map not found. Ensure preceding cells have run successfully.")

# --- 3. Plot Base Map and Bubbles ---
fig, ax = plt.subplots(figsize=(14, 12))
la_county_base_proj.plot(ax=ax, color='#E8E8E8', edgecolor='#FFFFFF', linewidth=0.5)

# Calculate marker sizes based on property count.
scaling_factor = 0.15
sizes = gdf_clusters_proj['count'] * scaling_factor

# Define a diverging colormap and normalization for the logerror.
colors_list = ["#56B4E9", "#F2F2F2", "#D55E00"]
cmap_okabe = mcolors.LinearSegmentedColormap.from_list("OkabeDiverging", colors_list)
vmin = gdf_clusters_proj['logerror_mean'].min()
vmax = gdf_clusters_proj['logerror_mean'].max()
norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax) if vmin < 0 < vmax else plt.Normalize(vmin=vmin, vmax=vmax)

# Plot the bubble markers.
scatter = ax.scatter(
    x=gdf_clusters_proj.geometry.x,
    y=gdf_clusters_proj.geometry.y,
    s=sizes,
    c=gdf_clusters_proj['logerror_mean'],
    cmap='OkabeDiverging',
    norm=norm,
    alpha=0.85,
    edgecolors='black',
    linewidth=0.8,
    zorder=5
)
ax.set_xticks([])
ax.set_yticks([])

# --- 4. Legends ---
# Colorbar for 'logerror_mean'.
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.2)
cbar = fig.colorbar(scatter, cax=cax, orientation='vertical')
cbar.set_label('Mean Directional Error (logerror)', fontsize=24, labelpad=35)
cbar.ax.tick_params(labelsize=24)

# Size Legend for 'count'.
min_c, max_c, mid_c = gdf_clusters_proj['count'].min(), gdf_clusters_proj['count'].max(), gdf_clusters_proj['count'].mean()
legend_values = [int(min_c), int(mid_c), int(max_c)]
handles = [ax.scatter([], [], s=v * scaling_factor, c='#999999', alpha=0.8, edgecolors='white', linewidth=0.8) for v in legend_values]
size_legend = ax.legend(
    handles,
    [f'{v:,}' for v in legend_values],
    title="Properties\nper Cluster",
    loc="upper right",
    bbox_to_anchor=(-0.02, 1.0),
    frameon=True,
    facecolor='white',
    edgecolor='#CCCCCC',
    labelspacing=1.2,
    borderpad=1.0,
    handletextpad=1.5,
    fontsize=24
)
size_legend.get_frame().set_linewidth(0.6)
plt.setp(size_legend.get_title(), fontsize=24, fontweight='bold')

plt.tight_layout(pad=2.0)
for _ax in plt.gcf().axes:
    enforce_academic_formatting(_ax)
plt.savefig('spatial_bubble_map.pdf', format='pdf', bbox_inches='tight')
plt.savefig('spatial_bubble_map.svg', format='svg', bbox_inches='tight')
plt.show()


## 10. Comparative Analysis: Cluster Migration


In [ ]:
# =================================================
# 10.1 SLOPEGRAPH: CLUSTER MIGRATION
# =================================================
# This cell creates a slopegraph to visualize how properties migrate between clusters when the fairness weighting is applied.

# --- 1. Data Preparation ---
# Load the full datasets for both baseline and weighted runs.
baseline_full_df = cached_data.get("geo_gdf_mapped_Baseline")
weighted_full_df = cached_data.get("geo_gdf_mapped_Weighted (ADI x2.0)")

if baseline_full_df is not None and weighted_full_df is not None:
    if 'latitude' in baseline_full_df.columns and 'latitude' in weighted_full_df.columns:
        # Select relevant columns and merge dataframes on property location to track migration.
        baseline_clusters = baseline_full_df[['latitude', 'longitude', 'clusters', 'ADI_STATERNK']].rename(columns={'clusters': 'source_node'})
        weighted_clusters = weighted_full_df[['latitude', 'longitude', 'clusters', 'ADI_STATERNK']].rename(columns={'clusters': 'target_node'})
        merged_df = pd.merge(baseline_clusters, weighted_clusters, on=['latitude', 'longitude'], suffixes=('_src', '_tgt'))

        # Calculate median ADI for source and target clusters to position them on the y-axis.
        source_adi = merged_df.groupby('source_node')['ADI_STATERNK_src'].median()
        target_adi = merged_df.groupby('target_node')['ADI_STATERNK_tgt'].median()

        # Count the number of properties flowing from each source to each target cluster.
        flow_df = merged_df.groupby(['source_node', 'target_node']).size().reset_index(name='property_count')

        # --- 2. Coordinate Setup ---
        fig, ax = plt.subplots(figsize=(16, 9))
        x_source, x_target = 0, 1

        # --- 3. Node Positioning & Axis Drawing ---
        y_source_map = source_adi.to_dict()
        y_target_map = target_adi.to_dict()
        ax.plot([x_source, x_source], [1, 10], color='black', linewidth=1.5)
        ax.plot([x_target, x_target], [1, 10], color='black', linewidth=1.5)
        ax.text(x_source, 0.5, "Baseline\nClustering", ha='center', va='bottom', fontsize=26, fontweight='bold')
        ax.text(x_target, 0.5, "Optimized Weighted\nClustering", ha='center', va='bottom', fontsize=26, fontweight='bold')

        # --- 4. Line Drawing ---
        # Draw lines between source and target clusters, with line width proportional to property count.
        max_count = flow_df['property_count'].max()
        max_linewidth = 25.0

        # Highlight flows into the most extreme over/undervalued clusters.
        highlight_nodes = [24, 6]
        flow_df['is_highlighted'] = flow_df['target_node'].isin(highlight_nodes)
        flow_df = flow_df.sort_values(by='is_highlighted') # Draw highlighted lines last

        for _, row in flow_df.iterrows():
            src, tgt = row['source_node'], row['target_node']
            if src in y_source_map and tgt in y_target_map:
                y0 = y_source_map[src]
                y1 = y_target_map[tgt]
                lw = max(0.5, (row['property_count'] / max_count) * max_linewidth)

                if tgt == 24: # Overvalued
                    color, alpha, zorder = '#D55E00', 1.0, 3
                elif tgt == 6: # Undervalued
                    color, alpha, zorder = '#56B4E9', 1.0, 3
                else: # Other flows
                    color, alpha, zorder = '#999999', 0.3, 1

                ax.plot([x_source, x_target], [y0, y1], color=color, linewidth=lw, alpha=alpha, zorder=zorder)

        # --- 5. Formatting & Export ---
        ax.set_xlim(-0.5, 1.5)
        ax.set_ylim(0.5, 10.5)
        ax.invert_yaxis() # Place ADI 1 (affluent) at the top.

        ax.set_yticks(range(1, 11))
        ax.set_yticklabels([str(i) for i in range(1, 11)], fontsize=24)
        ax.set_xticks([])
        ax.set_ylabel("Median Area Deprivation Index (ADI)", fontsize=26, labelpad=35)

        for _ax in plt.gcf().axes:
            enforce_academic_formatting(_ax)
        plt.savefig('hyperparameter_slopegraph.pdf', format='pdf', bbox_inches='tight')
        plt.savefig('hyperparameter_slopegraph.svg', format='svg', bbox_inches='tight')
        plt.show()
    else:
        print("⚠️ 'latitude'/'longitude' columns not found.")
else:
    print("⚠️ Required data for slopegraph not found.")


## 11. Narrative Extension Graphs


In [ ]:
# =================================================
# 11.1 Predicted vs. Actual Price Scatter Plot
# =================================================
# This cell plots the AVM's predicted price against the actual price, colored by ADI.
df = cached_data.get("socio_df_mapped_Baseline")

if df is not None:
    price_col = 'taxvaluedollarcnt' if 'taxvaluedollarcnt' in df.columns else None
    if price_col is None:
        print("Could not find a price column (like 'taxvaluedollarcnt'). Please check your data.")
    else:
        fig, ax = plt.subplots(figsize=(12, 8))

        # Calculate predicted price using the logerror formula: logerror = log(Pred) - log(Actual)
        df['predicted_price'] = df[price_col] * np.exp(df['logerror'])

        # Sample the data to avoid overplotting and improve performance.
        sample_df = df.sample(n=min(5000, len(df)), random_state=42)

        scatter = plt.scatter(sample_df[price_col], sample_df['predicted_price'],
                              c=sample_df['rounded_adi'], cmap='OkabeDiverging', alpha=0.5, s=20)

        # Add a 45-degree line representing a perfect prediction.
        max_val = min(sample_df[price_col].max(), sample_df['predicted_price'].max())
        plt.plot([0, max_val], [0, max_val], 'k--', lw=2, label='Perfect Prediction (45°)')

        cbar = plt.colorbar(scatter)
        cbar.set_label('Area Deprivation Index (ADI)', fontsize=22)
        cbar.ax.tick_params(labelsize=18)

        plt.xlabel('Actual Property Value', fontsize=22)
        plt.ylabel('AVM Predicted Value', fontsize=22)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)

        # Use a log scale to better visualize data with a wide range of values.
        plt.xscale('log')
        plt.yscale('log')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=20, frameon=True, edgecolor='black', framealpha=1)
        plt.tight_layout(pad=2.0)
        for _ax in plt.gcf().axes:
            enforce_academic_formatting(_ax)
        plt.savefig('predicted_vs_actual_scatter.pdf', bbox_inches='tight')
        plt.savefig('predicted_vs_actual_scatter.svg', bbox_inches='tight')
        plt.show()
else:
    print("Data not loaded.")


In [ ]:
# =================================================
# 11.2 Absolute vs. Directional Error Bar Chart (The "Blind Spot" Graph)
# =================================================
# This cell illustrates how similar absolute errors can hide very different directional biases.
df_summary = cached_data.get("geo_summary_Weighted (ADI x2.0)")

if df_summary is not None:
    # Filter out small clusters to ensure robustness.
    valid_clusters = df_summary[df_summary['count'] > 100]

    # Manually select the extreme clusters identified in the paper for comparison.
    target_clusters = valid_clusters[valid_clusters['c'].isin([6, 24])]

    if len(target_clusters) == 2:
        fig, ax = plt.subplots(figsize=(13, 6))

        x = np.arange(2)
        width = 0.35

        abs_errs = target_clusters['abs_logerror_mean'].values
        dir_errs = target_clusters['logerror_mean'].values
        labels = [f"Cluster {int(c)} (ADI ~{adi})" for c, adi in zip(target_clusters['c'], target_clusters['rounded_adi'])]

        # Plot absolute error (magnitude) and directional error (bias).
        ax.bar(x - width/2, abs_errs, width, label='Absolute Error (Magnitude)', color='gray')
        colors = ['#D55E00' if val > 0 else '#56B4E9' for val in dir_errs]
        ax.bar(x + width/2, dir_errs, width, label='Directional Error (logerror)', color=colors)

        ax.set_ylabel('Mean Error', fontsize=22)
        ax.tick_params(axis='y', labelsize=18)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=20)

        # Create a custom legend to explain the colors.
        import matplotlib.patches as mpatches
        gray_patch = mpatches.Patch(color='gray', label='Absolute Error')
        over_patch = mpatches.Patch(color='#D55E00', label='Overvaluation (Positive)')
        under_patch = mpatches.Patch(color='#56B4E9', label='Undervaluation (Negative)')
        ax.legend(handles=[gray_patch, over_patch, under_patch], fontsize=20, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, edgecolor='black', framealpha=1)

        plt.axhline(0, color='black', linewidth=1)
        plt.tight_layout(pad=2.0)
        for _ax in plt.gcf().axes:
            enforce_academic_formatting(_ax)
        plt.savefig('absolute_vs_directional_blindspot.pdf', bbox_inches='tight')
        plt.savefig('absolute_vs_directional_blindspot.svg', bbox_inches='tight')
        plt.show()
    else:
        print("Clusters 6 and 24 not found. Please adjust the target_clusters filter.")
else:
    print("Summary data not loaded.")


In [ ]:
# =================================================
# 11.3 Structural Variance Violin Plots (Affluent vs. Deprived)
# =================================================
# This cell compares the distribution of property sizes between the most affluent and most deprived areas.
df = cached_data.get("socio_df_mapped_Baseline")

if df is not None:
    struct_col = 'calculatedfinishedsquarefeet'
    if struct_col in df.columns:
        fig, ax = plt.subplots(figsize=(10, 6))

        # Filter to only the most affluent (ADI 1) and most deprived (ADI 10) groups.
        extreme_df = df[df['rounded_adi'].isin([1, 10])].copy()
        extreme_df['ADI_Group'] = extreme_df['rounded_adi'].map({1.0: 'Affluent (ADI 1)', 10.0: 'Deprived (ADI 10)'})

        sns.violinplot(x='ADI_Group', y=struct_col, data=extreme_df, palette=['#56B4E9', '#D55E00'])

        plt.xlabel('')
        plt.ylabel('Calculated Finished Square Feet', fontsize=22)
        plt.xticks(fontsize=20)
        plt.yticks(fontsize=18)

        # Limit y-axis to remove extreme outliers for better visual clarity.
        upper_limit = extreme_df[struct_col].quantile(0.99)
        plt.ylim(0, upper_limit)

        plt.tight_layout(pad=2.0)
        for _ax in plt.gcf().axes:
            enforce_academic_formatting(_ax)
        plt.savefig('structural_variance_violin.pdf', format='pdf', bbox_inches='tight')
        plt.savefig('structural_variance_violin.svg', format='svg', bbox_inches='tight')
        plt.show()
    else:
        print(f"Column {struct_col} not found in data.")
else:
    print("Data not loaded.")


In [ ]:
# =================================================
# 11.4 De-unified Bivariate Maps (Geography vs. Deprivation)
# =================================================
# This cell creates two separate hexbin maps: one for ADI and one for logerror, to show their individual spatial patterns.
gdf = cached_data.get("geo_gdf_mapped_Weighted (ADI x2.0)")

if gdf is not None and not gdf.empty:
    # Map 1: Median Area Deprivation Index (ADI)
    fig_adi, ax_adi = plt.subplots(figsize=(10, 10))
    hb1 = ax_adi.hexbin(gdf.geometry.x, gdf.geometry.y, C=gdf['cluster_dominant_adi'].round(),
                         gridsize=50, cmap='OkabeSequential', reduce_C_function=np.median)
    ax_adi.set_xticks([])
    ax_adi.set_yticks([])
    cb1 = fig_adi.colorbar(hb1, ax=ax_adi, orientation='horizontal', pad=0.05)
    cb1.ax.tick_params(labelsize=18)
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('bivariate_leakage_map_adi.pdf', bbox_inches='tight')
    plt.savefig('bivariate_leakage_map_adi.svg', bbox_inches='tight')
    plt.show()

    # Map 2: Mean Logerror (Bias)
    import matplotlib.colors as mcolors
    div_norm = mcolors.TwoSlopeNorm(vmin=-0.1, vcenter=0., vmax=0.1)

    fig_err, ax_err = plt.subplots(figsize=(10, 10))
    hb2 = ax_err.hexbin(gdf.geometry.x, gdf.geometry.y, C=gdf['logerror'],
                         gridsize=50, cmap='OkabeDiverging', norm=div_norm, reduce_C_function=np.mean)
    ax_err.set_xticks([])
    ax_err.set_yticks([])
    cb2 = fig_err.colorbar(hb2, ax=ax_err, orientation='horizontal', pad=0.05)
    cb2.ax.tick_params(labelsize=18)
    plt.tight_layout(pad=2.0)
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('bivariate_leakage_map_logerror.pdf', bbox_inches='tight')
    plt.savefig('bivariate_leakage_map_logerror.svg', bbox_inches='tight')
    plt.show()
else:
    print("GeoDataFrame not loaded.")


In [ ]:
# =================================================
# 11.5 Error by Price Decile (Regressivity Plot)
# =================================================
# This cell investigates AVM regressivity by plotting mean error against property value deciles.
df = cached_data.get("socio_df_mapped_Baseline")

if df is not None:
    price_col = 'taxvaluedollarcnt' if 'taxvaluedollarcnt' in df.columns else None
    if price_col is not None:
        # Create 10 price deciles (1 = cheapest, 10 = most expensive).
        df['Price_Decile'] = pd.qcut(df[price_col], 10, labels=False) + 1

        # Calculate the mean logerror for each decile.
        decile_stats = df.groupby('Price_Decile')['logerror'].mean().reset_index()

        plt.figure(figsize=(10, 6))
        ax = sns.barplot(x='Price_Decile', y='logerror', data=decile_stats, palette='OkabeDiverging')

        plt.xlabel('Property Value Decile (1 = Cheapest, 10 = Most Expensive)', fontsize=22)
        plt.ylabel('Mean Directional Error (logerror)', fontsize=22)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)

        plt.axhline(0, color='black', linewidth=1)
        plt.tight_layout(pad=2.0)
        for _ax in plt.gcf().axes:
            enforce_academic_formatting(_ax)
        plt.savefig('regressivity_decile_plot.pdf', format='pdf', bbox_inches='tight')
        plt.savefig('regressivity_decile_plot.svg', format='svg', bbox_inches='tight')
        plt.show()
    else:
        print("Price column not found.")
else:
    print("Data not loaded.")


In [ ]:
# =================================================
# 11.6 Diverging Bar Chart (Directional Harm by Cluster)
# =================================================
# This cell creates a diverging bar chart to show the directional bias of every geographic cluster, sorted by ADI.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

run_name = "Weighted (ADI x2.0)"
df_summary = cached_data.get(f"geo_summary_{run_name}")

if df_summary is not None:
    # Sort clusters by ADI so the plot has a clean socioeconomic progression.
    df_plot = df_summary.copy()
    df_plot = df_plot.sort_values(by=['dominant_adi', 'logerror_mean'], ascending=[False, True])

    # Create descriptive labels for each cluster.
    df_plot['Cluster_Label'] = df_plot.apply(lambda row: f"C{int(row['c'])} (ADI: {row['dominant_adi']:.1f})", axis=1)

    # Assign colors based on the direction of the error (overvaluation vs. undervaluation).
    df_plot['Color'] = df_plot['logerror_mean'].apply(lambda x: "#D55E00" if x > 0 else "#56B4E9")

    plt.figure(figsize=(14, 16))
    ax = plt.gca()

    # Plot the horizontal diverging bars.
    bars = ax.barh(df_plot['Cluster_Label'], df_plot['logerror_mean'],
                   color=df_plot['Color'], edgecolor='white', height=0.75, zorder=3)

    # Draw a thick baseline at zero error.
    ax.axvline(0, color='black', linewidth=2.5, linestyle='-', zorder=5)

    # Add data labels at the end of each bar for precise readability.
    for bar in bars:
        width = bar.get_width()
        label_x_pos = width + 0.001 if width > 0 else width - 0.001
        ha = 'left' if width > 0 else 'right'
        ax.text(label_x_pos, bar.get_y() + bar.get_height()/2, f"{width:+.3f}",
                ha=ha, va='center', fontsize=16, fontweight='bold', color=bar.get_facecolor())

    # Add extra space on the x-axis to prevent labels from being cut off.
    x_min, x_max = ax.get_xlim()
    ax.set_xlim(x_min - 0.006, x_max + 0.005)

    # Titles and Labels.
    ax.set_xlabel('Mean Directional Logerror (Bias)', fontsize=24, labelpad=25)
    ax.set_ylabel('Geographic Clusters (Affluent → Deprived)', fontsize=24, labelpad=25)
    ax.tick_params(axis='y', which='major', labelsize=16)
    ax.tick_params(axis='x', which='major', labelsize=20)

    # Apply formatting and save.
    if 'enforce_academic_formatting' in globals():
        enforce_academic_formatting(ax)
    plt.tight_layout(pad=2.0)
    plt.savefig('diverging_directional_harm.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('diverging_directional_harm.svg', format='svg', bbox_inches='tight')
    plt.show()
else:
    print(f"Required data missing for Diverging Bar Chart ({run_name}).")


## 12. Accessibility-Focused Geographic Map


In [ ]:
# =================================================
# 12.1 ACCESSIBILITY-FOCUSED GEOGRAPHIC MAP
# =================================================
# This cell creates a geographic map that uses distinct shapes and a colorblind-safe palette
# to differentiate between overvalued, undervalued, and other clusters, ensuring accessibility.

# --- Configuration ---
TARGET_RUN_NAME = "Weighted (ADI x2.0)"
NUM_CLUSTERS_TO_SHOW = 3
# --- End Configuration ---

# --- 1. Data Retrieval ---
# Retrieve the necessary data from the cache, processed in earlier steps.
gdf = cached_data.get(f"geo_gdf_mapped_{TARGET_RUN_NAME}")
df_summary = cached_data.get(f"geo_summary_{TARGET_RUN_NAME}")
la_border_orig = cached_data.get('la_border') # Use a different name to avoid CRS conflicts

# --- Guard Clause ---
if gdf is None or df_summary is None or la_border_orig is None:
    print(f"⚠️ Prerequisite data for '{TARGET_RUN_NAME}' not found. Please ensure preceding cells have run successfully.")
else:
    # --- 2. Identify Extreme Clusters ---
    # Find the top N overvalued and undervalued clusters based on mean logerror.
    df_geo_sorted = df_summary.sort_values(by='logerror_mean', ascending=False)
    top_overvalued = df_geo_sorted[df_geo_sorted['logerror_mean'] > 0].head(NUM_CLUSTERS_TO_SHOW)
    top_undervalued = df_geo_sorted[df_geo_sorted['logerror_mean'] < 0].sort_values(by='logerror_mean', ascending=True).head(NUM_CLUSTERS_TO_SHOW)

    overvalued_cids = top_overvalued['c'].tolist()
    undervalued_cids = top_undervalued['c'].tolist()
    print(f"✅ Identified critical clusters: {len(overvalued_cids)} overvalued, {len(undervalued_cids)} undervalued.")

    # --- 3. Plotting ---
    fig, ax = plt.subplots(1, 1, figsize=(15, 15))

    # Reproject layers to Web Mercator (EPSG:3857) for compatibility with contextily basemaps.
    gdf_proj = gdf.to_crs(epsg=3857)
    la_border_proj = la_border_orig.to_crs(epsg=3857)

    # a. Plot all non-critical clusters in a neutral gray.
    gdf_other = gdf_proj[~gdf_proj['clusters'].isin(overvalued_cids + undervalued_cids)]
    gdf_other.plot(ax=ax, color='grey', marker='.', markersize=15, alpha=0.3, label='Other Clusters')

    # b. Plot overvalued clusters using a distinct shape and color.
    gdf_over = gdf_proj[gdf_proj['clusters'].isin(overvalued_cids)]
    if not gdf_over.empty:
        gdf_over.plot(ax=ax, color=okabe_ito[5], marker='^', markersize=80, alpha=0.9, edgecolor='black', linewidth=0.8, label='Top Overvalued')

    # c. Plot undervalued clusters using another distinct shape and color.
    gdf_under = gdf_proj[gdf_proj['clusters'].isin(undervalued_cids)]
    if not gdf_under.empty:
        gdf_under.plot(ax=ax, color=okabe_ito[4], marker='v', markersize=80, alpha=0.9, edgecolor='black', linewidth=0.8, label='Top Undervalued')

    # d. Add the LA County border.
    la_border_proj.plot(ax=ax, color='none', edgecolor='black', linewidth=2)

    # --- 4. Final Touches & Formatting ---
    # Set explicit axis limits to ensure the entire area is visible.
    try:
        minx, miny, maxx, maxy = la_border_proj.total_bounds
        buffer = (maxx - minx) * 0.05 # 5% buffer
        ax.set_xlim(minx - buffer, maxx + buffer)
        ax.set_ylim(miny - buffer, maxy + buffer)
    except Exception as e:
        print(f"❌ Could not set plot bounds: {e}")

    # Add a basemap for geographic context.
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=10)
    ax.axis('off')

    # Create a clear, ordered legend outside the plot.
    handles, labels = ax.get_legend_handles_labels()
    try:
        order = [labels.index('Top Overvalued'), labels.index('Top Undervalued'), labels.index('Other Clusters')]
        ax.legend([handles[idx] for idx in order], [labels[idx] for idx in order],
                  title="Cluster Type", markerscale=2, fontsize=20, title_fontsize=22,
                  bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, edgecolor='black', framealpha=1)
    except ValueError:
        ax.legend(title="Cluster Type", markerscale=2, fontsize=20, title_fontsize=22,
                  bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, edgecolor='black', framealpha=1)

    plt.tight_layout()
    # Apply the standard academic formatting.
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('optimised_spatial_run_accessible.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('optimised_spatial_run_accessible.svg', format='svg', bbox_inches='tight')
    plt.show()


In [ ]:
# =================================================
# 1. IMPORTS & SETUP
# =================================================
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx

# =================================================
# 2. CONFIGURATION
# =================================================
DATA_DIR = r"C:\Users\kaiso\PycharmProjects\BPAI\clustering_results\Results_Unweighted\s42_Unweighted_Baseline"
FULL_DATA_FILENAME = "+REG+SEN+ERR_full_data.csv"
CENSUS_SHAPEFILE_PATH = r"C:\Users\kaiso\PycharmProjects\BPAI\Data\shapefile_2015_BlockGroup_Calif\tl_2015_06_bg.shp"

full_data_path = os.path.join(DATA_DIR, FULL_DATA_FILENAME)

# =================================================
# 3. DATA LOADING & PREPARATION
# =================================================
try:
    la_border = cached_data.get('la_border')
    if la_border is not None:
        la_border = la_border.to_crs(epsg=3857)
        print("✅ LA County border loaded from cache.")
    else:
        print("❌ Could not load LA County border from cache.")
        la_border = None
except Exception as e:
    print(f"❌ Error processing cached border: {e}")
    la_border = None

if os.path.exists(full_data_path):
    df_full = pd.read_csv(full_data_path)
    gdf = gpd.GeoDataFrame(
        df_full,
        geometry=gpd.points_from_xy(df_full.longitude, df_full.latitude),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)
    print(f"✅ Full geographic data loaded with {len(df_full)} properties.")
else:
    gdf = None
    print(f"❌ CRITICAL: Full data not found at {full_data_path}")

# =================================================
# 4. PLOTTING
# =================================================
if gdf is not None and not gdf.empty:
    fig, ax = plt.subplots(1, 1, figsize=(15, 15))

    # Use a categorical colormap from Okabe-Ito
    unique_ranks = sorted(gdf['ADI_STATERNK'].unique())
    palette = sns.color_palette(okabe_ito, len(unique_ranks))
    color_map = {rank: palette[i] for i, rank in enumerate(unique_ranks)}

    for rank, color in color_map.items():
        gdf[gdf['ADI_STATERNK'] == rank].plot(ax=ax, color=color, marker='.', markersize=15, alpha=0.7, label=f'ADI Rank {rank}')

    if la_border is not None:
        la_border.plot(ax=ax, color='none', edgecolor='black', linewidth=2)

    try:
        minx, miny, maxx, maxy = gdf.total_bounds
        buffer = (maxx - minx) * 0.05
        ax.set_xlim(minx - buffer, maxx + buffer)
        ax.set_ylim(miny - buffer, maxy + buffer)
        print(f"✅ Set plot bounds to: ({minx:.2f}, {miny:.2f}, {maxx:.2f}, {maxy:.2f})")
    except Exception as e:
        print(f"❌ Could not set plot bounds: {e}")

    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=10)

    ax.axis('off')
    ax.legend(title="ADI Rank", markerscale=2, fontsize=24, bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, edgecolor='black', framealpha=1)

    plt.tight_layout()
    for _ax in plt.gcf().axes:
        enforce_academic_formatting(_ax)
    plt.savefig('geographic_run_map.pdf', format='pdf', bbox_inches='tight')
    plt.savefig('geographic_run_map.svg', format='svg', bbox_inches='tight')
    plt.show()

else:
    print("Could not generate plot because data was not loaded.")